# LeetCode 544 — Output Contest Matches bằng Python

Link đề: https://leetcode.com/problems/output-contest-matches/ · Độ khó: **Medium** · Chủ đề: Recursion, String, Simulation (mô phỏng)

Skeleton LeetCode:

```python
class Solution:
    def findContestMatch(self, n: int) -> str:
        ...
```

Ghi chú: đây là bài **Premium** (cần tài khoản trả phí trên LeetCode). Đề và các ví dụ trong notebook được viết theo hiểu biết về bài, nên hãy **đối chiếu lại với đề gốc** nếu bạn có tài khoản Premium. Mọi đáp án đều đã được kiểm chứng bằng code (so với một bộ kiểm tra tính hợp lệ độc lập và so giữa các cách giải). Các ví dụ `n = 2` và `n = 16` là ví dụ **tự dựng** (đối chiếu lại đề gốc nếu cần).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có `n` đội (hạng `1` mạnh nhất, hạng `n` yếu nhất), mỗi vòng **đội mạnh nhất còn lại ghép với đội yếu nhất còn lại**, rồi in ra toàn bộ sơ đồ ghép cặp dưới dạng **chuỗi có ngoặc và dấu phẩy**, ví dụ `((1,4),(2,3))`.
2. Nắm các công cụ Python: `f-string` / nối chuỗi, chỉ số âm và công thức đối xứng `n - 1 - i`, toán tử dịch bit `>>`, `list comprehension`, đệ quy, `ast.literal_eval`, `bit_length`.
3. Đi từ **mô phỏng từng vòng với list mới** → **mô phỏng tại chỗ trên một mảng** → **đệ quy từ trên xuống theo hạng hạt giống** → **thứ tự hạt giống + đếm số ngoặc bằng bit** → **tuple lồng nhau để Python tự in**.
4. Hiểu mấu chốt: sau mỗi vòng, mỗi "cặp" đóng vai **một đội mới** có hạng bằng hạng của đội mạnh hơn trong cặp, nên **quy tắc ghép `i` với `m-1-i` lặp lại y nguyên** ở vòng sau, và cần đúng `log2(n)` vòng.
5. Tránh các lỗi kinh điển: ghép **hai đội liền kề** thay vì hai đội đối xứng, lệch chỉ số `n-i` thay vì `n-1-i`, ghi đè sai chỗ khi làm tại chỗ, và in thừa dấu cách sau dấu phẩy.

## 2. Đề bài & ví dụ

Trong vòng playoffs NBA, người ta thường cho đội **khá mạnh** đấu với đội **khá yếu** (ví dụ đội hạng `1` đấu đội hạng thứ `n`) để trận đấu hấp dẫn hơn.

Cho `n` đội, được đánh số từ `1` đến `n` theo **hạng ban đầu** (hạng `1` mạnh nhất, hạng `n` yếu nhất). Hãy trả về **sơ đồ các trận đấu cuối cùng** dưới dạng một **chuỗi**.

Ta dùng dấu ngoặc `(` `)` để biểu diễn **một cặp đấu** và dấu phẩy `,` để **ngăn cách** hai bên của cặp. Ở **mỗi vòng**, luôn ghép đội mạnh hơn với đội yếu hơn (mạnh nhất còn lại với yếu nhất còn lại, mạnh thứ hai với yếu thứ hai, ...).

**Ví dụ 1**

- Input: `n = 4`
- Output: `"((1,4),(2,3))"`
- Giải thích: vòng 1 ghép `1` với `4`, `2` với `3` được `(1,4),(2,3)`. Vòng 2: hai đội thắng của `(1,4)` và `(2,3)` đấu với nhau nên ta bọc thêm ngoặc ngoài cùng: `((1,4),(2,3))`.

**Ví dụ 2**

- Input: `n = 8`
- Output: `"(((1,8),(4,5)),((2,7),(3,6)))"`
- Giải thích: vòng 1: `(1,8),(2,7),(3,6),(4,5)`; vòng 2: `((1,8),(4,5)),((2,7),(3,6))`; vòng 3 (chung kết) bọc thêm một lớp ngoặc.

**Ràng buộc:**

- `n == 2^x` với `x` nằm trong khoảng `[1, 12]` (tức `n` là lũy thừa của `2`, từ `2` đến `4096`).

Chú ý: sau vòng cuối chỉ còn **một** cặp duy nhất, nên chuỗi kết quả luôn bắt đầu bằng `(` và kết thúc bằng `)`. **Không** có dấu cách trong chuỗi.

In [1]:
from typing import List, Optional
from collections import deque
import ast
import random

examples = [
    (4, "((1,4),(2,3))"),
    (8, "(((1,8),(4,5)),((2,7),(3,6)))"),
]
extra_examples = [  # tự dựng, đối chiếu lại đề gốc nếu cần
    (2, "(1,2)"),
    (16, "((((1,16),(8,9)),((4,13),(5,12))),(((2,15),(7,10)),((3,14),(6,11))))"),
]
for n, expected in examples + extra_examples:
    print("n =", n, "-> kỳ vọng", expected)

n = 4 -> kỳ vọng ((1,4),(2,3))
n = 8 -> kỳ vọng (((1,8),(4,5)),((2,7),(3,6)))
n = 2 -> kỳ vọng (1,2)
n = 16 -> kỳ vọng ((((1,16),(8,9)),((4,13),(5,12))),(((2,15),(7,10)),((3,14),(6,11))))


## 3. Kiến thức nền

### 3.1. Chuỗi có ngoặc bằng `f-string` và phép cộng chuỗi

`f"({a},{b})"` ghép hai giá trị `a`, `b` vào một chuỗi. Trong `f-string`, **dấu ngoặc tròn `(` `)` là ký tự bình thường**, còn **dấu ngoặc nhọn `{}`** mới là chỗ chèn biến. Cũng có thể nối bằng `+`: `"(" + a + "," + b + ")"`. Cả hai cho cùng kết quả, `f-string` ngắn gọn hơn.

In [2]:
a, b = "1", "4"
print(f"({a},{b})")                 # (1,4)
print("(" + a + "," + b + ")")      # (1,4)
x, y = f"({a},{b})", "(2,3)"
print(f"({x},{y})")                 # ((1,4),(2,3))  <- bọc thêm một lớp

(1,4)
(1,4)
((1,4),(2,3))


### 3.2. Chỉ số đối xứng: phần tử thứ `i` và phần tử đối xứng với nó

Với list có `m` phần tử (chỉ số `0..m-1`), phần tử **đối xứng** với chỉ số `i` ở đầu bên kia là chỉ số `m - 1 - i`. Ví dụ `m = 8`: `0↔7`, `1↔6`, `2↔5`, `3↔4`. Python cho phép chỉ số âm: `s[-1]` là phần tử cuối, `s[-1-i]` chính là `s[m-1-i]`. Đây là công thức "mạnh nhất ghép yếu nhất" của bài.

In [3]:
s = ["1", "2", "3", "4", "5", "6", "7", "8"]
m = len(s)
for i in range(m // 2):
    print(i, "<->", m - 1 - i, "|", s[i], s[m - 1 - i], "| chỉ số âm:", s[-1 - i])

0 <-> 7 | 1 8 | chỉ số âm: 8
1 <-> 6 | 2 7 | chỉ số âm: 7
2 <-> 5 | 3 6 | chỉ số âm: 6
3 <-> 4 | 4 5 | chỉ số âm: 5


### 3.3. Dịch bit `>>` và chia đôi số nguyên

`n >> 1` dịch các bit của `n` sang phải một vị trí, tức là **chia nguyên cho `2`** (`n // 2`). `n >>= 1` là viết tắt của `n = n >> 1`. Vì `n` là lũy thừa của `2`, chia đôi liên tục sẽ rơi chính xác về `1`: `8 → 4 → 2 → 1`. Số lần chia đôi là `log2(n)`, cũng là **số vòng đấu**.

In [4]:
n = 8
rounds = 0
while n > 1:
    print("còn", n, "đội -> có", n >> 1, "cặp")
    n >>= 1
    rounds += 1
print("số vòng:", rounds)          # 3

print((4096).bit_length() - 1)     # 12 : log2 của lũy thừa 2 = bit_length() - 1

còn 8 đội -> có 4 cặp
còn 4 đội -> có 2 cặp
còn 2 đội -> có 1 cặp
số vòng: 3
12


### 3.4. `list comprehension` tạo danh sách `"1", "2", ..., "n"`

`[str(i + 1) for i in range(n)]` tạo list các chuỗi `"1"` đến `"n"`. Ta dùng **chuỗi** (không dùng số) vì từ vòng 1 trở đi mỗi phần tử sẽ là cả một cặp như `"(1,4)"`.

In [5]:
n = 8
s = [str(i + 1) for i in range(n)]
print(s)
print([i + 1 for i in range(n)])     # list số, để so sánh

['1', '2', '3', '4', '5', '6', '7', '8']
[1, 2, 3, 4, 5, 6, 7, 8]


### 3.5. Đệ quy: hàm tự gọi chính nó

Một hàm đệ quy có **điều kiện dừng** (trường hợp nhỏ nhất, trả luôn) và **bước đệ quy** (đưa bài toán về bài nhỏ hơn). Ví dụ đơn giản: `sum_to(k) = k + sum_to(k - 1)`, dừng ở `k = 0`. Với bài này, ta sẽ định nghĩa "chuỗi của một đội ở vòng đầu là chính số hạng của nó" (điều kiện dừng) và "chuỗi của một đội ở vòng sau là cặp của hai chuỗi con" (bước đệ quy).

In [6]:
def sum_to(k):
    if k == 0:                # điều kiện dừng
        return 0
    return k + sum_to(k - 1)  # bước đệ quy


print(sum_to(4))   # 10

10


### 3.6. `ast.literal_eval`: biến chuỗi ngoặc thành tuple lồng nhau

Chuỗi `"((1,4),(2,3))"` chính là cú pháp của một **tuple lồng nhau** trong Python. `ast.literal_eval` đọc an toàn một chuỗi như vậy thành đối tượng thật. Ta sẽ dùng nó để **kiểm tra độc lập** đáp án (không cần tự viết bộ phân tích cú pháp).

In [7]:
t = ast.literal_eval("((1,4),(2,3))")
print(t, type(t))           # ((1, 4), (2, 3)) <class 'tuple'>
print(t[0], t[1], t[0][1])  # (1, 4) (2, 3) 4
print(str(t))               # ((1, 4), (2, 3))  <- Python tự thêm dấu cách sau phẩy!
print(str(t).replace(" ", ""))

((1, 4), (2, 3)) <class 'tuple'>
(1, 4) (2, 3) 4
((1, 4), (2, 3))
((1,4),(2,3))


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `n = 8`. Ta tự làm từng vòng bằng tay.

**Vòng 1.** Có 8 đội, xếp theo hạng `1 2 3 4 5 6 7 8`. Ghép mạnh nhất với yếu nhất: `1-8`, `2-7`, `3-6`, `4-5`. Ta viết `(1,8) (2,7) (3,6) (4,5)`.

**Câu hỏi then chốt:** ở vòng 2, ai đấu với ai? Giả sử đội mạnh hơn luôn thắng, thì bốn đội thắng của vòng 1 là `1, 2, 3, 4`. Ta lại ghép mạnh nhất với yếu nhất: `1-4`, `2-3`. Nghĩa là **cặp `(1,8)` đấu cặp `(4,5)`**, và **cặp `(2,7)` đấu cặp `(3,6)`**.

Điều quan trọng: ta không cần biết kết quả thật. Chỉ cần coi mỗi cặp là **một "siêu đội"** có hạng bằng hạng của đội mạnh hơn trong cặp. Bốn siêu đội `(1,8) (2,7) (3,6) (4,5)` đã **nằm sẵn theo thứ tự hạng** `1,2,3,4`. Vì vậy quy tắc "**phần tử `i` ghép phần tử `m-1-i`**" lặp lại nguyên xi:

| Vòng | Danh sách (mỗi phần tử là một "siêu đội", đã xếp theo hạng) | Số phần tử `m` | Ghép |
|---|---|---|---|
| 0 | `1 2 3 4 5 6 7 8` | 8 | `0-7, 1-6, 2-5, 3-4` |
| 1 | `(1,8) (2,7) (3,6) (4,5)` | 4 | `0-3, 1-2` |
| 2 | `((1,8),(4,5)) ((2,7),(3,6))` | 2 | `0-1` |
| 3 | `(((1,8),(4,5)),((2,7),(3,6)))` | 1 | dừng |

Mỗi vòng, số phần tử **giảm một nửa**, nên có `log2(8) = 3` vòng.

In [8]:
teams = [str(i) for i in range(1, 9)]
print("vòng 0:", teams)
m = len(teams)
r1 = [f"({teams[i]},{teams[m - 1 - i]})" for i in range(m // 2)]
print("vòng 1:", r1)
m = len(r1)
r2 = [f"({r1[i]},{r1[m - 1 - i]})" for i in range(m // 2)]
print("vòng 2:", r2)
m = len(r2)
r3 = [f"({r2[i]},{r2[m - 1 - i]})" for i in range(m // 2)]
print("vòng 3:", r3)
assert r3 == ["(((1,8),(4,5)),((2,7),(3,6)))"]

vòng 0: ['1', '2', '3', '4', '5', '6', '7', '8']
vòng 1: ['(1,8)', '(2,7)', '(3,6)', '(4,5)']
vòng 2: ['((1,8),(4,5))', '((2,7),(3,6))']
vòng 3: ['(((1,8),(4,5)),((2,7),(3,6)))']


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng vòng, tạo list mới

**Ý tưởng.** Làm đúng như đã tính tay:

1. `cur = ["1", "2", ..., "n"]`.
2. Khi `cur` còn nhiều hơn một phần tử: tạo `nxt` rỗng; với mỗi `i` trong `0 .. len(cur)//2 - 1`, thêm `"(" + cur[i] + "," + cur[-1-i] + ")"` vào `nxt`; rồi gán `cur = nxt`.
3. Khi chỉ còn một phần tử, đó là đáp án.

Đây là cách "ngây thơ" nhất vì ta **cấp phát một list mới ở mỗi vòng** và **sao chép lại chuỗi** nhiều lần.

In [9]:
def contest_rounds_demo(n):
    """Cách 1 (bản in từng vòng để quan sát)."""
    cur = [str(i) for i in range(1, n + 1)]
    print("vòng 0:", cur)
    r = 0
    while len(cur) > 1:
        nxt = []
        for i in range(len(cur) // 2):
            nxt.append("(" + cur[i] + "," + cur[len(cur) - 1 - i] + ")")
        cur = nxt
        r += 1
        print(f"vòng {r}:", cur)
    return cur[0]


print(contest_rounds_demo(8))

vòng 0: ['1', '2', '3', '4', '5', '6', '7', '8']
vòng 1: ['(1,8)', '(2,7)', '(3,6)', '(4,5)']
vòng 2: ['((1,8),(4,5))', '((2,7),(3,6))']
vòng 3: ['(((1,8),(4,5)),((2,7),(3,6)))']
(((1,8),(4,5)),((2,7),(3,6)))


In [10]:
def contest_new_list(n):
    """Cách 1: baseline, tạo list mới mỗi vòng."""
    cur = [str(i) for i in range(1, n + 1)]
    while len(cur) > 1:
        nxt = []
        for i in range(len(cur) // 2):
            nxt.append("(" + cur[i] + "," + cur[len(cur) - 1 - i] + ")")
        cur = nxt
    return cur[0]


for n, expected in examples + extra_examples:
    got = contest_new_list(n)
    print(n, got == expected, got)

4 True ((1,4),(2,3))
8 True (((1,8),(4,5)),((2,7),(3,6)))
2 True (1,2)
16 True ((((1,16),(8,9)),((4,13),(5,12))),(((2,15),(7,10)),((3,14),(6,11))))


**Độ phức tạp.**

- Số vòng: `log2(n)`. Ở mỗi vòng ta duyệt `m/2` cặp và sao chép chuỗi; **tổng độ dài các chuỗi** trong list luôn xấp xỉ `4n` ký tự chữ số (mỗi số tối đa `4` chữ số) cộng thêm ngoặc/phẩy. Vì vậy mỗi vòng tốn `O(n)` ký tự sao chép, tổng `O(n log n)`.
- Bộ nhớ phụ: list hiện tại + list mới ⇒ `O(n)`.

**Nút thắt.** Cách này **đã đủ nhanh** với `n ≤ 4096` (đúng độ phức tạp chuẩn của bài). "Nút thắt" ở đây là về **độ gọn**: tạo list mới mỗi vòng. Phần tiếp theo cho thấy ta có thể **dùng lại chính mảng cũ** (không cấp phát list mới), và có những cách nhìn khác hẳn (từ trên xuống, theo thứ tự lá).

## 6. Ý tưởng thuật toán / bước đột phá

Có **ba nhận xét** dẫn tới các cách làm tiếp theo:

1. **Ghi đè tại chỗ.** Ở vòng có `m` phần tử, ta chỉ **đọc cả hai nửa** nhưng chỉ **ghi vào nửa đầu** (`s[0 .. m/2-1]`). Phần tử `s[i]` (`i < m/2`) được ghi bằng cặp `(s[i], s[m-1-i])`; chỉ số `m-1-i ≥ m/2` nằm ở nửa sau, **chưa bị ghi đè**. Vậy ta chỉ cần **một mảng** và biến `m` giảm dần.
2. **Nhìn từ trên xuống.** Đội ở "vị trí `i`" khi còn `m` siêu đội sẽ ở vòng trước thành cặp `(i, 2m+1-i)` gồm đội `i` và đội đối xứng của nó trong `2m` siêu đội (`1-based`). Ta có công thức đệ quy: `build(i, m) = "(" + build(i, 2m) + "," + build(2m+1-i, 2m) + ")"`, dừng khi `m = n` (trả `str(i)`). Đáp án là `build(1, 1)`.
3. **Chỉ cần biết thứ tự lá.** Sơ đồ là một **cây nhị phân đầy đủ** có `n` lá. Thứ tự các hạng của lá từ trái sang phải được sinh bằng một dãy: bắt đầu `[1]`, mỗi lần nhân đôi, thay mỗi `s` bởi `s, m+1-s`. Số dấu ngoặc mở trước lá thứ `j` và số ngoặc đóng sau nó có thể tính **bằng bit** (số bit `0` ở cuối của `j` hoặc `j+1`), rồi nối một lần bằng `join`.

Ba nhận xét này sinh ra Cách 2, Cách 3, Cách 4. Cách 5 chỉ là thủ thuật Python: dùng **tuple lồng nhau** và để Python in ra.

## 7. Các cách tiếp theo

### Cách 2 — Ghi đè tại chỗ trên một mảng (cách chuẩn)

**Giải thích từng đoạn.**

- `s = [str(i + 1) for i in range(n)]`: mảng ban đầu.
- `while n > 1:` n chính là số phần tử đang "sống" của `s` (m ở phần trước).
- `for i in range(n >> 1):` duyệt `n/2` cặp. Cặp thứ `i` gồm `s[i]` và `s[n-i-1]`.
- `s[i] = f"({s[i]},{s[n-i-1]})"`: ghi cặp vào nửa đầu. Vì `i < n/2 ≤ n-i-1` nên `s[n-i-1]` chưa bị ghi đè.
- `n >>= 1`: sang vòng sau, chỉ còn nửa đầu là "sống".
- `return s[0]`.

In [11]:
def contest_inplace_demo(n):
    s = [str(i + 1) for i in range(n)]
    print("ban đầu:", s)
    while n > 1:
        for i in range(n >> 1):
            s[i] = f"({s[i]},{s[n - i - 1]})"
        n >>= 1
        print("n =", n, "phần còn sống:", s[:n])
    return s[0]


print(contest_inplace_demo(8))

ban đầu: ['1', '2', '3', '4', '5', '6', '7', '8']
n = 4 phần còn sống: ['(1,8)', '(2,7)', '(3,6)', '(4,5)']
n = 2 phần còn sống: ['((1,8),(4,5))', '((2,7),(3,6))']
n = 1 phần còn sống: ['(((1,8),(4,5)),((2,7),(3,6)))']
(((1,8),(4,5)),((2,7),(3,6)))


In [12]:
class Solution2:
    def findContestMatch(self, n: int) -> str:
        s = [str(i + 1) for i in range(n)]
        while n > 1:
            for i in range(n >> 1):
                s[i] = f"({s[i]},{s[n - i - 1]})"
            n >>= 1
        return s[0]


for n, expected in examples + extra_examples:
    assert Solution2().findContestMatch(n) == expected
print("Cách 2 đúng trên các ví dụ")

Cách 2 đúng trên các ví dụ


**Độ phức tạp Cách 2:** Time `O(n log n)` (mỗi vòng sao chép `O(n)` ký tự), Space `O(n)` (một mảng `n` chuỗi, không cấp phát list mới mỗi vòng).

---

### Cách 3 — Đệ quy từ trên xuống theo hạng hạt giống

**Giải thích từng đoạn.**

- Hàm `build(i, m)` trả chuỗi của "siêu đội hạng `i`" **khi đang có `m` siêu đội** (`1 ≤ i ≤ m`).
- Nếu `m == n` thì đó là một đội thật, trả `str(i)` (điều kiện dừng).
- Ngược lại, ở vòng trước (có `2m` siêu đội), siêu đội hạng `i` là **cặp** của đội hạng `i` và đội hạng `2m + 1 - i`. Nên `build(i, m) = "(" + build(i, 2m) + "," + build(2m+1-i, 2m) + ")"`.
- Gọi `build(1, 1)`: "một siêu đội duy nhất" = cả giải.
- Độ sâu đệ quy chỉ `log2(n) ≤ 12`, rất an toàn.

In [13]:
def contest_topdown_demo(n):
    def build(i, m, depth=0):
        print("  " * depth + f"build(i={i}, m={m})")
        if m == n:
            return str(i)
        left = build(i, 2 * m, depth + 1)
        right = build(2 * m + 1 - i, 2 * m, depth + 1)
        return "(" + left + "," + right + ")"

    return build(1, 1)


print(contest_topdown_demo(4))

build(i=1, m=1)
  build(i=1, m=2)
    build(i=1, m=4)
    build(i=4, m=4)
  build(i=2, m=2)
    build(i=2, m=4)
    build(i=3, m=4)
((1,4),(2,3))


In [14]:
class Solution3:
    def findContestMatch(self, n: int) -> str:
        def build(i: int, m: int) -> str:
            if m == n:
                return str(i)
            return "(" + build(i, 2 * m) + "," + build(2 * m + 1 - i, 2 * m) + ")"

        return build(1, 1)


for n, expected in examples + extra_examples:
    assert Solution3().findContestMatch(n) == expected
print("Cách 3 đúng trên các ví dụ")

Cách 3 đúng trên các ví dụ


**Độ phức tạp Cách 3:** có `2n - 1` lời gọi; mỗi lời gọi nối chuỗi con nên tổng chi phí sao chép `O(n log n)`. Space `O(n)` cho chuỗi kết quả cộng `O(log n)` ngăn xếp đệ quy.

---

### Cách 4 — Thứ tự hạt giống + đếm ngoặc bằng bit (một lần `join`)

Sơ đồ là một **cây nhị phân đầy đủ** có `n` lá, chiều cao `d = log2(n)`. Chuỗi kết quả đọc các lá từ trái sang phải, và quanh mỗi lá có một số ngoặc `(` đứng trước, `)` đứng sau.

**Bước 1: thứ tự hạng của các lá.** Bắt đầu `order = [1]`. Mỗi lần "nhân đôi" lên `m` phần tử (`m = 2, 4, ..., n`), thay mỗi `s` bằng hai phần tử `s, m + 1 - s`. Ví dụ `[1] → [1,2] → [1,4,2,3] → [1,8,4,5,2,7,3,6]` (đây đúng là các số xuất hiện khi đọc `(((1,8),(4,5)),((2,7),(3,6)))` theo thứ tự).

**Bước 2: số ngoặc mở trước lá `j` (0-based).** Lá `j` là **lá trái nhất** của một số nút trong. Với `j = 0`: là lá trái nhất của **cả `d` tổ tiên**. Với `j > 0`: bằng số bit `0` ở cuối biểu diễn nhị phân của `j` (ví dụ `j = 4 = 100₂` có `2`). Viết gọn: `(j & -j).bit_length() - 1`.

**Bước 3: số ngoặc đóng sau lá `j`.** Lá `j` là **lá phải nhất** của các nút trong tương ứng với số bit `0` ở cuối của `j + 1`. Với `j = n - 1` thì `j + 1 = n = 2^d` có đúng `d` bit `0` cuối, nên không cần trường hợp riêng.

**Bước 4:** mỗi lá cho một đoạn `"(" * opens + str(seed) + ")" * closes`; nối các đoạn bằng dấu phẩy.

In [15]:
def trailing_zeros(x):
    return (x & -x).bit_length() - 1     # chỉ dùng cho x > 0


print([(j, bin(j), trailing_zeros(j)) for j in range(1, 9)])

# Thứ tự hạt giống của các lá
order = [1]
m = 1
while m < 8:
    m *= 2
    order = [x for s in order for x in (s, m + 1 - s)]
    print("m =", m, "order =", order)

[(1, '0b1', 0), (2, '0b10', 1), (3, '0b11', 0), (4, '0b100', 2), (5, '0b101', 0), (6, '0b110', 1), (7, '0b111', 0), (8, '0b1000', 3)]
m = 2 order = [1, 2]
m = 4 order = [1, 4, 2, 3]
m = 8 order = [1, 8, 4, 5, 2, 7, 3, 6]


In [16]:
def contest_seed_order_demo(n):
    d = n.bit_length() - 1
    order = [1]
    m = 1
    while m < n:
        m *= 2
        order = [x for s in order for x in (s, m + 1 - s)]
    print("d =", d, "order =", order)
    parts = []
    for j, seed in enumerate(order):
        opens = d if j == 0 else trailing_zeros(j)
        closes = trailing_zeros(j + 1)
        print(f"  j={j} seed={seed} mở={opens} đóng={closes}")
        parts.append("(" * opens + str(seed) + ")" * closes)
    return ",".join(parts)


print(contest_seed_order_demo(8))

d = 3 order = [1, 8, 4, 5, 2, 7, 3, 6]
  j=0 seed=1 mở=3 đóng=0
  j=1 seed=8 mở=0 đóng=1
  j=2 seed=4 mở=1 đóng=0
  j=3 seed=5 mở=0 đóng=2
  j=4 seed=2 mở=2 đóng=0
  j=5 seed=7 mở=0 đóng=1
  j=6 seed=3 mở=1 đóng=0
  j=7 seed=6 mở=0 đóng=3
(((1,8),(4,5)),((2,7),(3,6)))


In [17]:
class Solution4:
    def findContestMatch(self, n: int) -> str:
        d = n.bit_length() - 1
        order = [1]
        m = 1
        while m < n:
            m *= 2
            order = [x for s in order for x in (s, m + 1 - s)]
        parts = []
        for j, seed in enumerate(order):
            opens = d if j == 0 else (j & -j).bit_length() - 1
            closes = ((j + 1) & -(j + 1)).bit_length() - 1
            parts.append("(" * opens + str(seed) + ")" * closes)
        return ",".join(parts)


for n, expected in examples + extra_examples:
    assert Solution4().findContestMatch(n) == expected
print("Cách 4 đúng trên các ví dụ")

Cách 4 đúng trên các ví dụ


**Độ phức tạp Cách 4:** sinh `order` tốn `O(n)` (`1 + 2 + 4 + ... + n`), tính ngoặc `O(1)` mỗi lá (thao tác bit), nối chuỗi một lần ⇒ **Time `O(n)`** (độ dài đáp án cũng `O(n)` vì số chữ số tối đa `4`). Space `O(n)`. Đây là cách **nhanh nhất về mặt lý thuyết**, nhưng khó đọc hơn.

---

### Cách 5 — Tuple lồng nhau và để Python tự in

**Ý tưởng.** Thay vì ghép chuỗi, giữ các "siêu đội" là **số** hoặc **tuple**: `(1, 4)`, `((1, 8), (4, 5))`... Cuối cùng `str(tuple)` cho ra `"((1, 4), (2, 3))"`, chỉ khác là Python **thêm dấu cách sau mỗi dấu phẩy**; ta xóa chúng bằng `.replace(" ", "")`.

Cách này ngắn nhưng **vẫn là mô phỏng như Cách 1**, chỉ đổi cách biểu diễn dữ liệu. Ta giữ nó như một cách viết gọn và để dễ kiểm tra bằng mắt.

In [18]:
class Solution5:
    def findContestMatch(self, n: int) -> str:
        cur = list(range(1, n + 1))
        while len(cur) > 1:
            cur = [(cur[i], cur[-1 - i]) for i in range(len(cur) // 2)]
        return str(cur[0]).replace(" ", "")


for n, expected in examples + extra_examples:
    assert Solution5().findContestMatch(n) == expected
print("Cách 5 đúng trên các ví dụ")
print(Solution5().findContestMatch(8))

Cách 5 đúng trên các ví dụ
(((1,8),(4,5)),((2,7),(3,6)))


**Độ phức tạp Cách 5:** Time `O(n log n)` (giống Cách 1) vì `str` của tuple lồng nhau chỉ gọi một lần ở cuối, Space `O(n)`.

## 8. TRACE từng bước

**Bảng trạng thái cho Cách 2 với `n = 8`** (mảng `s`, chỉ ghi nửa còn sống sau mỗi vòng):

| Vòng | `n` (số đang sống) | `i` | Ghép `s[i]` với `s[n-1-i]` | `s` sau khi ghi (nửa đầu) |
|---|---|---|---|---|
| bắt đầu | 8 | | | `1 2 3 4 5 6 7 8` |
| 1 | 8 | 0 | `1` với `8` | `(1,8) 2 3 4 …` |
| 1 | 8 | 1 | `2` với `7` | `(1,8) (2,7) 3 4 …` |
| 1 | 8 | 2 | `3` với `6` | `(1,8) (2,7) (3,6) 4 …` |
| 1 | 8 | 3 | `4` với `5` | `(1,8) (2,7) (3,6) (4,5) …` |
| 2 | 4 | 0 | `(1,8)` với `(4,5)` | `((1,8),(4,5)) (2,7) …` |
| 2 | 4 | 1 | `(2,7)` với `(3,6)` | `((1,8),(4,5)) ((2,7),(3,6)) …` |
| 3 | 2 | 0 | cả hai | `(((1,8),(4,5)),((2,7),(3,6))) …` |

Dấu `…` là phần nửa sau đã "chết", không dùng nữa. Hàm `trace_inplace` dưới đây in đúng các bước đó cho mảng đầy đủ.

In [19]:
def trace_inplace(n):
    s = [str(i + 1) for i in range(n)]
    print("ban đầu:", s)
    vong = 0
    while n > 1:
        vong += 1
        print(f"--- vòng {vong}: n = {n} (còn {n} phần tử sống)")
        for i in range(n >> 1):
            a, b = s[i], s[n - i - 1]
            s[i] = f"({a},{b})"
            print(f"  i={i}: ghép s[{i}]={a} với s[{n - i - 1}]={b} -> s[{i}] = {s[i]}")
        n >>= 1
        print("  mảng:", s)
    print("kết quả:", s[0])
    return s[0]


trace_inplace(8)

ban đầu: ['1', '2', '3', '4', '5', '6', '7', '8']
--- vòng 1: n = 8 (còn 8 phần tử sống)
  i=0: ghép s[0]=1 với s[7]=8 -> s[0] = (1,8)
  i=1: ghép s[1]=2 với s[6]=7 -> s[1] = (2,7)
  i=2: ghép s[2]=3 với s[5]=6 -> s[2] = (3,6)
  i=3: ghép s[3]=4 với s[4]=5 -> s[3] = (4,5)
  mảng: ['(1,8)', '(2,7)', '(3,6)', '(4,5)', '5', '6', '7', '8']
--- vòng 2: n = 4 (còn 4 phần tử sống)
  i=0: ghép s[0]=(1,8) với s[3]=(4,5) -> s[0] = ((1,8),(4,5))
  i=1: ghép s[1]=(2,7) với s[2]=(3,6) -> s[1] = ((2,7),(3,6))
  mảng: ['((1,8),(4,5))', '((2,7),(3,6))', '(3,6)', '(4,5)', '5', '6', '7', '8']
--- vòng 3: n = 2 (còn 2 phần tử sống)
  i=0: ghép s[0]=((1,8),(4,5)) với s[1]=((2,7),(3,6)) -> s[0] = (((1,8),(4,5)),((2,7),(3,6)))
  mảng: ['(((1,8),(4,5)),((2,7),(3,6)))', '((2,7),(3,6))', '(3,6)', '(4,5)', '5', '6', '7', '8']
kết quả: (((1,8),(4,5)),((2,7),(3,6)))


'(((1,8),(4,5)),((2,7),(3,6)))'

**Trace Cách 4** cho `n = 8`: thứ tự lá là `[1,8,4,5,2,7,3,6]`, `d = 3`. Bảng mở/đóng ngoặc:

| `j` | seed | mở (số bit 0 cuối của `j`, riêng `j=0` là `d`) | đóng (số bit 0 cuối của `j+1`) | đoạn |
|---|---|---|---|---|
| 0 | 1 | 3 | 0 | `(((1` |
| 1 | 8 | 0 | 1 | `8)` |
| 2 | 4 | 1 | 0 | `(4` |
| 3 | 5 | 0 | 2 | `5))` |
| 4 | 2 | 2 | 0 | `((2` |
| 5 | 7 | 0 | 1 | `7)` |
| 6 | 3 | 1 | 0 | `(3` |
| 7 | 6 | 0 | 3 | `6)))` |

Nối bằng dấu phẩy: `(((1,8),(4,5)),((2,7),(3,6)))`.

In [20]:
def trace_seed_order(n):
    d = n.bit_length() - 1
    order = [1]
    m = 1
    while m < n:
        m *= 2
        order = [x for s in order for x in (s, m + 1 - s)]
    print(f"{'j':>3} {'seed':>5} {'mở':>3} {'đóng':>5}  đoạn")
    parts = []
    for j, seed in enumerate(order):
        opens = d if j == 0 else trailing_zeros(j)
        closes = trailing_zeros(j + 1)
        piece = "(" * opens + str(seed) + ")" * closes
        print(f"{j:>3} {seed:>5} {opens:>3} {closes:>5}  {piece}")
        parts.append(piece)
    res = ",".join(parts)
    print("kết quả:", res)
    return res


assert trace_seed_order(8) == "(((1,8),(4,5)),((2,7),(3,6)))"

  j  seed  mở  đóng  đoạn
  0     1   3     0  (((1
  1     8   0     1  8)
  2     4   1     0  (4
  3     5   0     2  5))
  4     2   2     0  ((2
  5     7   0     1  7)
  6     3   1     0  (3
  7     6   0     3  6)))
kết quả: (((1,8),(4,5)),((2,7),(3,6)))


## 9. Code nộp LeetCode

Ta nộp **Cách 2** (ghi đè tại chỗ): ngắn, dễ nhớ, đúng độ phức tạp `O(n log n)` và không có rủi ro về đệ quy hay bit. Trên LeetCode không cần `import` gì thêm.

In [21]:
class Solution:
    def findContestMatch(self, n: int) -> str:
        s = [str(i + 1) for i in range(n)]
        while n > 1:
            for i in range(n >> 1):
                s[i] = f"({s[i]},{s[n - i - 1]})"
            n >>= 1
        return s[0]


for n, expected in examples + extra_examples:
    assert Solution().findContestMatch(n) == expected
print("Code nộp đúng trên 4 ví dụ")

Code nộp đúng trên 4 ví dụ


## 10. Vì sao thuật toán đúng

**Bất biến (invariant).** Trước mỗi vòng có `m` phần tử "sống" (`m` là lũy thừa của `2`), `s[0..m-1]` thỏa: `s[k]` là chuỗi sơ đồ của "siêu đội" đứng thứ `k+1` về độ mạnh, tức là siêu đội có **đội mạnh nhất bên trong có hạng gốc nhỏ thứ `k+1`** trong các siêu đội (nói cách khác, các siêu đội **được xếp theo hạng tăng dần**).

- **Khởi đầu:** `m = n`, `s[k] = str(k+1)`, đúng.
- **Duy trì:** vòng này ghép `s[i]` với `s[m-1-i]` cho `i = 0 .. m/2-1`. Đây đúng là "mạnh nhất với yếu nhất, mạnh thứ hai với yếu thứ hai..." theo đề. Cặp mới có đội mạnh nhất chính là đội mạnh nhất của `s[i]` (vì `i < m-1-i` nên `s[i]` mạnh hơn). Cặp `i` mới vì vậy có hạng thứ `i`, nghĩa là **thứ tự `0,1,...,m/2-1` vẫn giữ nguyên theo hạng**. Bất biến đúng với `m/2`.
- **An toàn khi ghi đè:** vòng này ghi vào `s[i]` (`i < m/2`) và đọc `s[m-1-i]` (`m-1-i ≥ m/2`), hai vùng **rời nhau** nên đọc không bao giờ gặp giá trị đã bị ghi mới.
- **Kết thúc:** khi `m = 1` chỉ còn một siêu đội gồm cả `n` đội, `s[0]` là chuỗi của cả sơ đồ. Vì `n` là lũy thừa của `2` nên `m` luôn chẵn cho đến khi bằng `1` (không bao giờ có "đội lẻ").

**Về giả thiết "đội mạnh hơn thắng":** chuỗi kết quả chỉ phụ thuộc cách **ghép cặp theo hạng**, không phụ thuộc kết quả thật của trận đấu. Ta quy ước người thắng của mỗi cặp là đội có hạng nhỏ hơn, nên mỗi siêu đội mang hạng của đội mạnh hơn trong cặp, đúng như bất biến ở trên.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. List mới mỗi vòng | `O(n log n)` | `O(n)` | `log n` vòng, mỗi vòng sao chép `O(n)` ký tự |
| 2. Ghi đè tại chỗ (nộp) | `O(n log n)` | `O(n)` | như trên nhưng một mảng |
| 3. Đệ quy từ trên xuống | `O(n log n)` | `O(n)` + `O(log n)` ngăn xếp | `2n-1` lời gọi |
| 4. Thứ tự hạt giống + bit | `O(n)` | `O(n)` | một lần `join`, không sao chép lại |
| 5. Tuple lồng nhau | `O(n log n)` | `O(n)` | giống Cách 1, đổi biểu diễn |

Với `n ≤ 4096` đáp án dài `27 562` ký tự. Mọi cách đều chạy tức thì, nên chọn cách **dễ viết và dễ đúng nhất**. Lưu ý độ dài đáp án cũng là cận dưới `Ω(n)` nên `O(n)` ở Cách 4 là tối ưu.

## 12. Bộ test

### 12.1. Ví dụ đề, biên và thuộc tính

Vì mỗi `n` chỉ có **một** đáp án đúng, ta so sánh trực tiếp. Dưới đây gom năm cách vào cùng một bảng và so với **bộ kiểm tra tính hợp lệ độc lập** `is_valid_bracket`: bộ này dùng `ast.literal_eval` để đọc chuỗi thành cây rồi kiểm tra, **không dùng bất kỳ cách giải nào ở trên**.

Tính chất mà `is_valid_bracket` kiểm tra (chiều cao nút trong `h`, lá có chiều cao `0`):

- Chuỗi không có dấu cách, đọc được thành tuple lồng nhau; mọi nút trong có đúng hai con, hai con **cùng chiều cao** (cây đầy đủ), tập lá là `{1..n}` không trùng.
- Gọi `a`, `b` là hạng nhỏ nhất trong hai cây con của một nút chiều cao `h`: phải có `a < b` và `a + b = n / 2^(h-1) + 1` (đúng quy tắc "mạnh nhất ghép yếu nhất").

In [22]:
def is_valid_bracket(n, s):
    """Oracle độc lập: kiểm tra chuỗi s có đúng là sơ đồ của n đội hay không."""
    if " " in s:
        return False
    try:
        t = ast.literal_eval(s) if n > 1 else int(s)
    except (ValueError, SyntaxError):
        return False
    seen = []

    def walk(node):
        """Trả (hạng nhỏ nhất, chiều cao) của cây con; ném lỗi nếu sai."""
        if isinstance(node, int):
            seen.append(node)
            return node, 0
        if not (isinstance(node, tuple) and len(node) == 2):
            raise ValueError("nút không phải cặp")
        a, ha = walk(node[0])
        b, hb = walk(node[1])
        if ha != hb:
            raise ValueError("hai con khác chiều cao")
        h = ha + 1
        if not (a < b and a + b == n // 2 ** (h - 1) + 1):
            raise ValueError("ghép sai theo hạng")
        return a, h

    try:
        _, height = walk(t)
    except ValueError:
        return False
    return height == n.bit_length() - 1 and sorted(seen) == list(range(1, n + 1))


assert is_valid_bracket(4, "((1,4),(2,3))")
assert not is_valid_bracket(4, "((1,2),(3,4))")        # ghép liền kề
assert not is_valid_bracket(4, "((1, 4),(2, 3))")      # có dấu cách
assert not is_valid_bracket(4, "((1,4),(3,2))")        # sai thứ tự mạnh trước
assert is_valid_bracket(1, "1")
print("Oracle tự kiểm tra OK")

Oracle tự kiểm tra OK


In [23]:
solvers = {
    "Cách 1 list mới": contest_new_list,
    "Cách 2 tại chỗ": Solution2().findContestMatch,
    "Cách 3 đệ quy": Solution3().findContestMatch,
    "Cách 4 bit": Solution4().findContestMatch,
    "Cách 5 tuple": Solution5().findContestMatch,
    "Code nộp": Solution().findContestMatch,
}

# Ví dụ đề + tự dựng
for n, expected in examples + extra_examples:
    for name, f in solvers.items():
        assert f(n) == expected, (name, n)

# Toàn bộ n hợp lệ: 2^1 .. 2^12
for x in range(1, 13):
    n = 2 ** x
    results = {name: f(n) for name, f in solvers.items()}
    assert len(set(results.values())) == 1, n
    ans = results["Code nộp"]
    assert is_valid_bracket(n, ans), n
    assert ans.count("(") == n - 1 and ans.count(")") == n - 1 and ans.count(",") == n - 1
    print(f"n = {n:>4}: độ dài {len(ans):>6}, bắt đầu {ans[:12]!r}")
print("Tất cả 12 giá trị n đều khớp giữa 5 cách + oracle")

n =    2: độ dài      5, bắt đầu '(1,2)'
n =    4: độ dài     13, bắt đầu '((1,4),(2,3)'
n =    8: độ dài     29, bắt đầu '(((1,8),(4,5'
n =   16: độ dài     68, bắt đầu '((((1,16),(8'
n =   32: độ dài    148, bắt đầu '(((((1,32),('
n =   64: độ dài    308, bắt đầu '((((((1,64),'
n =  128: độ dài    657, bắt đầu '(((((((1,128'
n =  256: độ dài   1425, bắt đầu '((((((((1,25'
n =  512: độ dài   2961, bắt đầu '(((((((((1,5'
n = 1024: độ dài   6058, bắt đầu '((((((((((1,'


n = 2048: độ dài  13226, bắt đầu '(((((((((((1'


n = 4096: độ dài  27562, bắt đầu '(((((((((((('
Tất cả 12 giá trị n đều khớp giữa 5 cách + oracle


### 12.2. Phản ví dụ cho các cách làm sai

Kiểm tra rằng bộ kiểm tra **bắt được** những cách làm sai điển hình (để chắc là test của ta đủ mạnh):

In [24]:
def wrong_adjacent(n):      # ghép hai đội LIỀN KỀ
    s = [str(i + 1) for i in range(n)]
    while len(s) > 1:
        s = [f"({s[2 * i]},{s[2 * i + 1]})" for i in range(len(s) // 2)]
    return s[0]


def wrong_spaces(n):        # có dấu cách sau phẩy
    s = [str(i + 1) for i in range(n)]
    while len(s) > 1:
        s = [f"({s[i]}, {s[-1 - i]})" for i in range(len(s) // 2)]
    return s[0]


def wrong_pair_once(n):     # chỉ ghép vòng 1 rồi dừng (quên lặp các vòng sau)
    s = [str(i + 1) for i in range(n)]
    return ",".join(f"({s[i]},{s[-1 - i]})" for i in range(n // 2))


for f in (wrong_adjacent, wrong_spaces, wrong_pair_once):
    got = f(8)
    print(f.__name__, "->", got, "| hợp lệ?", is_valid_bracket(8, got))
    assert not is_valid_bracket(8, got)

wrong_adjacent -> (((1,2),(3,4)),((5,6),(7,8))) | hợp lệ? False
wrong_spaces -> (((1, 8), (4, 5)), ((2, 7), (3, 6))) | hợp lệ? False
wrong_pair_once -> (1,8),(2,7),(3,6),(4,5) | hợp lệ? False


### 12.3. Đối chiếu ngẫu nhiên có seed

Tập `n` hợp lệ chỉ có `12` giá trị nên ngẫu nhiên chủ yếu để chọn **thứ tự** kiểm tra và kiểm tra thêm các **tính chất** tổng quát: (a) hạng gặp nhau ở vòng 1 luôn có tổng `n + 1`; (b) hạng `1` luôn nằm đầu chuỗi và hạng `2` nằm ở cuối chuỗi (vì đội 2 ở nửa phải của cây).

In [25]:
rng = random.Random(544)
for _ in range(60):
    n = 2 ** rng.randint(1, 12)
    ans = Solution().findContestMatch(n)
    assert ans == Solution4().findContestMatch(n) == Solution3().findContestMatch(n)
    assert is_valid_bracket(n, ans)
    assert ans.startswith("(" * (n.bit_length() - 1) + "1,")    # hạng 1 đứng đầu, ghép với hạng n ở vòng 1
    t = ast.literal_eval(ans)

    def first_round(node):
        """Lấy tất cả cặp ở vòng 1 (cặp mà cả hai phần tử đều là số)."""
        if isinstance(node[0], int):
            return [node]
        return first_round(node[0]) + first_round(node[1])

    pairs = first_round(t)
    assert len(pairs) == n // 2 and all(a + b == n + 1 for a, b in pairs)
print("Ngẫu nhiên (seed 544) đạt")

Ngẫu nhiên (seed 544) đạt


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Ghép hai đội **liền kề** (`s[2i]` với `s[2i+1]`)

Nhiều người nghĩ "ghép từng đôi một" như bốc thăm. Nhưng đề yêu cầu **mạnh nhất với yếu nhất**.

In [26]:
def sai_lien_ke(n):
    s = [str(i + 1) for i in range(n)]
    while len(s) > 1:
        s = [f"({s[2 * i]},{s[2 * i + 1]})" for i in range(len(s) // 2)]
    return s[0]


print("sai :", sai_lien_ke(4))                     # ((1,2),(3,4))
print("đúng:", Solution().findContestMatch(4))     # ((1,4),(2,3))

sai : ((1,2),(3,4))
đúng: ((1,4),(2,3))


**Vì sao sai:** `(1,2)` ghép đội mạnh nhất với đội mạnh thứ hai. Phải ghép `s[i]` với `s[m-1-i]`, tức phần tử đối xứng ở **đầu bên kia** của danh sách. **Sửa:** dùng `s[-1 - i]` hoặc `s[m - 1 - i]`.

### Lỗi 2: Lệch chỉ số `n - i` thay vì `n - i - 1`

Chỉ số hợp lệ là `0..n-1`, nên đội đối xứng của `i` là `n - 1 - i`. Viết `n - i` sẽ lệch một vị trí (và với `i = 0` còn vượt biên).

In [27]:
def sai_lech_chi_so(n):
    s = [str(i + 1) for i in range(n)]
    while n > 1:
        for i in range(n >> 1):
            s[i] = f"({s[i]},{s[n - i]})"          # SAI: phải là n - i - 1
        n >>= 1
    return s[0]


try:
    print(sai_lech_chi_so(4))
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().findContestMatch(4))

IndexError: list index out of range
đúng: ((1,4),(2,3))


**Vì sao sai:** với `n = 4`, `i = 0` tra `s[4]` nhưng mảng chỉ có chỉ số `0..3` (khi `n` bằng kích thước mảng). Với các vòng sau (`n` nhỏ hơn độ dài mảng) thì **không báo lỗi** mà âm thầm lấy sai phần tử, nghĩa là có thể ra đáp án sai mà không có cảnh báo. **Sửa:** `s[n - i - 1]`; thử tay với `n = 2`: `i = 0` ghép `s[0]` và `s[1]`.

### Lỗi 3: Giảm `n` **trước** vòng lặp `for` (hoặc lặp sai số cặp)

`n >>= 1` phải làm **sau khi** đã ghép xong vòng hiện tại, vì công thức `s[n-i-1]` cần `n` cũ.

In [28]:
def sai_giam_n_som(n):
    s = [str(i + 1) for i in range(n)]
    while n > 1:
        n >>= 1                                    # SAI: giảm sớm
        for i in range(n):
            s[i] = f"({s[i]},{s[n - i - 1]})"
    return s[0]


print("sai :", sai_giam_n_som(8))
print("đúng:", Solution().findContestMatch(8))

sai : (((1,4),(2,3)),((1,4),(2,3)))
đúng: (((1,8),(4,5)),((2,7),(3,6)))


**Vì sao sai:** sau khi giảm sớm, `n` trở thành số cặp thay vì số phần tử; `s[n - i - 1]` lấy phần tử trong **nửa đầu** (đã/đang bị ghi đè) thay vì nửa sau. Kết quả có cặp ghép chéo lung tung. **Sửa:** `for i in range(n >> 1): ...` rồi mới `n >>= 1` (hoặc dùng biến phụ `half`).

### Lỗi 4: In thừa dấu cách / dùng `str(tuple)` mà quên xóa dấu cách

`str((1, 4))` cho `"(1, 4)"` có dấu cách sau phẩy, **không khớp** đáp án mong đợi `"(1,4)"`.

In [29]:
print(repr(str((1, 4))))                                  # '(1, 4)'  <- có dấu cách
print(repr(str(((1, 4), (2, 3))).replace(" ", "")))       # '((1,4),(2,3))'
assert str(((1, 4), (2, 3))) != "((1,4),(2,3))"
assert str(((1, 4), (2, 3))).replace(" ", "") == "((1,4),(2,3))"

'(1, 4)'
'((1,4),(2,3))'


**Sửa:** hoặc tự ghép chuỗi bằng `f"({a},{b})"` (không có dấu cách), hoặc nhớ `.replace(" ", "")`. LeetCode so sánh **từng ký tự**, nên thừa một dấu cách là sai.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "đội mạnh nhất ghép đội yếu nhất", "vòng loại trực tiếp", `n` là lũy thừa của `2`, kết quả là **cấu trúc lồng nhau** ⇒ nghĩ ngay tới **mô phỏng từng vòng** và **cây nhị phân đầy đủ**.
- **Hai đầu mảng ghép nhau:** `s[i]` với `s[m-1-i]` là mẫu hai con trỏ từ hai đầu (giống kiểm tra chuỗi đối xứng, Two Sum trên mảng đã sắp).
- **Mỗi vòng giảm một nửa ⇒ `log n` vòng.** Dùng `n >>= 1`.
- **Thu gọn thành một mảng:** nếu chỉ ghi nửa đầu và đọc nửa sau thì không cần mảng thứ hai.
- **Thử tay với `n = 4` và `n = 8` rồi mới viết code.** Hai ví dụ trong đề đủ để phát hiện lỗi ghép liền kề hoặc lệch chỉ số.
- **Kiểm tra bất biến nhanh:** ở vòng 1, tổng hai hạng trong mỗi cặp luôn là `n + 1`; ở vòng 2 là `n/2 + 1`, ...
- **Khi cần đáp án rất nhanh:** có công thức đóng dựa vào bit (Cách 4), nhưng đừng dùng nếu không cần.

## 15. Biến thể

1. **Trả về danh sách từng vòng** (không chỉ vòng cuối): dùng lại Cách 1 và lưu `cur` ở mỗi vòng.
2. **Đối thủ của hạng `i` ở vòng `r`** (giả sử đội mạnh hơn luôn thắng): nếu có `2^k` đội còn lại, đối thủ của `i` là `2^k + 1 - i`.
3. **Hai đội hạng `a`, `b` gặp nhau ở vòng nào?** Là vòng ứng với tổ tiên chung thấp nhất trong cây sơ đồ.
4. **`n` không phải lũy thừa của `2`:** cần cho các đội mạnh nhất "được miễn" vòng 1 (bye); phải thêm đội ảo cho đủ lũy thừa `2` gần nhất rồi bỏ qua trận có đội ảo.
5. **Đảo ngược bài toán:** cho chuỗi sơ đồ, tìm `n` (đếm số `,` cộng 1) hoặc kiểm tra sơ đồ có hợp lệ không (chính là `is_valid_bracket`).

In [30]:
def all_rounds(n):
    """Biến thể 1: trả về danh sách chuỗi của từng vòng."""
    cur = [str(i) for i in range(1, n + 1)]
    out = [cur]
    while len(cur) > 1:
        cur = [f"({cur[i]},{cur[-1 - i]})" for i in range(len(cur) // 2)]
        out.append(cur)
    return out


def opponent(n, seed, r):
    """Biến thể 2: đối thủ của hạng `seed` ở vòng r (1-based), nếu seed còn trụ tới vòng đó."""
    remaining = n >> (r - 1)           # số đội còn ở đầu vòng r
    assert 1 <= seed <= remaining
    return remaining + 1 - seed


rounds = all_rounds(8)
print(rounds)
print([opponent(8, 1, r) for r in (1, 2, 3)])   # [8, 4, 2]
print([opponent(8, 3, r) for r in (1, 2)])      # [6, 2]  (3 gặp 2 ở vòng 2 nếu thắng)

[['1', '2', '3', '4', '5', '6', '7', '8'], ['(1,8)', '(2,7)', '(3,6)', '(4,5)'], ['((1,8),(4,5))', '((2,7),(3,6))'], ['(((1,8),(4,5)),((2,7),(3,6)))']]
[8, 4, 2]
[6, 2]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. List mới mỗi vòng | Mô phỏng trực tiếp, mỗi vòng tạo list mới | `O(n log n)` | `O(n)` | Học nhập môn, dễ hiểu nhất |
| 2. Ghi đè tại chỗ | Một mảng, ghi nửa đầu, đọc nửa sau | `O(n log n)` | `O(n)` | **Nộp bài**, ngắn gọn, ít bộ nhớ phụ |
| 3. Đệ quy trên xuống | `build(i, m)` bọc cặp `(i, 2m+1-i)` | `O(n log n)` | `O(n)` + `O(log n)` | Thích tư duy cây, dễ mở rộng (in cây, đếm...) |
| 4. Thứ tự hạt giống + bit | Sinh thứ tự lá rồi đếm ngoặc bằng số bit 0 cuối | `O(n)` | `O(n)` | Muốn tối ưu tiệm cận / học thủ thuật bit |
| 5. Tuple lồng nhau | Giữ tuple, `str(...).replace(" ", "")` | `O(n log n)` | `O(n)` | Viết nhanh, kiểm tra bằng mắt |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Solution().findContestMatch(2)`; `len(Solution().findContestMatch(8))`; kết quả của `s = ["1","2","3","4"]` sau **một** lần chạy `for i in range(2): s[i] = f"({s[i]},{s[3 - i]})"`; và `16 >> 2`.

**Bài 2 (tính tay).** Với `n = 16`: viết ra **vòng 1** (8 cặp), và cho biết **đối thủ của hạng 1 ở mỗi vòng** (giả sử luôn thắng). Chuỗi cho `n = 16` có bao nhiêu dấu `(`, bao nhiêu dấu `,`?

**Bài 3 (điền TODO).** Hoàn thành `contest_todo(n)` theo Cách 2 (ghi đè tại chỗ).

In [31]:
def contest_todo(n):
    # TODO 1: s = list các chuỗi "1" .. "n"
    # TODO 2: while n > 1: với i trong range(n >> 1): s[i] = f"({s[i]},{s[n - i - 1]})"
    # TODO 3: n >>= 1 sau khi xong các cặp; cuối cùng return s[0]
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên chưa đúng).

**Bài 4 (tự viết lại).** Không nhìn code mẫu, viết `contest_topdown2(n)` theo cách đệ quy từ trên xuống (Cách 3), nhưng **bằng một hàm có ba tham số** `(lo_seed, hi_seed, size)` thay vì `build(i, m)`: gợi ý, cặp ở cấp trên là `(lo, hi)` với `lo + hi = size + 1`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `meeting_round(n, a, b)` trả về **vòng** (`1` = vòng đầu) mà hai hạng `a` và `b` **có thể gặp nhau** nếu đội mạnh hơn luôn thắng. Ví dụ `n = 8`: `meeting_round(8, 1, 8) = 1`, `meeting_round(8, 1, 4) = 2`, `meeting_round(8, 1, 2) = 3`. (b) Viết `parse_n(s)` nhận chuỗi sơ đồ, trả về `n`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"(1,2)"`; độ dài của `"(((1,8),(4,5)),((2,7),(3,6)))"` là `29` (8 số + 14 ngoặc + 7 phẩy); sau một lần chạy: `["(1,4)", "(2,3)", "3", "4"]`; `16 >> 2 = 4`. Bài 3: bám sát mục 9.
- **Mức 2 (vừa):** Bài 2: vòng 1 gồm `(1,16) (2,15) (3,14) (4,13) (5,12) (6,11) (7,10) (8,9)`; đối thủ của hạng 1: vòng 1 gặp `16`, vòng 2 gặp `8`, vòng 3 gặp `4`, vòng 4 gặp `2`. Số `(` bằng `n - 1 = 15`, số `,` cũng `n - 1 = 15`. Bài 4: nếu cặp cấp trên là `(lo, hi)` với size hiện tại là `size`, thì ở cấp dưới (size `2*size`), `lo` tách thành `(lo, 2*size + 1 - lo)` và `hi` tách thành `(hi, 2*size + 1 - hi)`.
- **Mức 3 (sâu):** Bài 5(a): hai đội `a` và `b` gặp nhau ở vòng `r` nếu chúng cùng thuộc một siêu đội ở vòng `r` nhưng **thuộc hai siêu đội con khác nhau** ở vòng `r-1`. Cách dễ: parse sơ đồ bằng `ast.literal_eval`, đi xuống cây; tại mỗi nút nếu `a` và `b` cùng nằm ở một con thì đi vào con đó, nếu tách thì chiều cao của nút hiện tại chính là vòng cần tìm. Bài 5(b): có `n` lá thì có `n - 1` dấu phẩy.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert Solution().findContestMatch(2) == "(1,2)"
assert len(Solution().findContestMatch(8)) == 29
s = ["1", "2", "3", "4"]
for i in range(2):
    s[i] = f"({s[i]},{s[3 - i]})"
assert s == ["(1,4)", "(2,3)", "3", "4"]
assert 16 >> 2 == 4

# Bài 2
ans16 = Solution().findContestMatch(16)
assert ans16 == extra_examples[1][1]
assert all_rounds(16)[1] == ["(1,16)", "(2,15)", "(3,14)", "(4,13)", "(5,12)", "(6,11)", "(7,10)", "(8,9)"]
assert [opponent(16, 1, r) for r in (1, 2, 3, 4)] == [16, 8, 4, 2]
assert ans16.count("(") == 15 and ans16.count(",") == 15


# Bài 3
def contest_todo_ans(n):
    s = [str(i + 1) for i in range(n)]
    while n > 1:
        for i in range(n >> 1):
            s[i] = f"({s[i]},{s[n - i - 1]})"
        n >>= 1
    return s[0]


for k in range(1, 13):
    assert contest_todo_ans(2 ** k) == Solution().findContestMatch(2 ** k)


# Bài 4
def contest_topdown2(n):
    def go(lo, hi, size):
        # lo, hi là hai hạng (1-based) đang đứng ở cấp có `size` siêu đội, lo + hi = size + 1
        if size == n:
            return f"({lo},{hi})"
        nxt = 2 * size
        return "(" + go(lo, nxt + 1 - lo, nxt) + "," + go(hi, nxt + 1 - hi, nxt) + ")"

    return "1" if n == 1 else go(1, 2, 2)


for k in range(1, 13):
    assert contest_topdown2(2 ** k) == Solution().findContestMatch(2 ** k), k


# Bài 5
def meeting_round(n, a, b):
    t = ast.literal_eval(Solution().findContestMatch(n))

    def flat(node):
        if isinstance(node, int):
            return {node}
        return flat(node[0]) | flat(node[1])

    h = n.bit_length() - 1
    node = t
    while True:
        left, right = node
        if a in flat(left) and b in flat(left):
            node, h = left, h - 1
        elif a in flat(right) and b in flat(right):
            node, h = right, h - 1
        else:
            return h


assert meeting_round(8, 1, 8) == 1
assert meeting_round(8, 1, 4) == 2
assert meeting_round(8, 1, 2) == 3
assert meeting_round(8, 3, 6) == 1 and meeting_round(8, 2, 3) == 2


def parse_n(s):
    return s.count(",") + 1


for k in range(1, 13):
    assert parse_n(Solution().findContestMatch(2 ** k)) == 2 ** k
print("Tất cả đáp án bài tập đều đúng.")

Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mô phỏng từng vòng, ghi đè tại chỗ)

```python
class Solution:
    def findContestMatch(self, n: int) -> str:
        s = [str(i + 1) for i in range(n)]      # mỗi phần tử là một "siêu đội"
        while n > 1:                            # n = số phần tử đang sống
            for i in range(n >> 1):             # n/2 cặp
                s[i] = f"({s[i]},{s[n - i - 1]})"   # mạnh nhất ghép yếu nhất
            n >>= 1                             # giảm một nửa SAU khi ghép xong
        return s[0]
```

**Checklist**

- [ ] Ghép `s[i]` với `s[n-1-i]` (đối xứng), không phải `s[i+1]`?
- [ ] Giảm `n` bằng `n >>= 1` **sau** vòng `for`, và lặp `n >> 1` cặp mỗi vòng?
- [ ] Phần tử là **chuỗi** (không phải số) để nối được ngoặc?
- [ ] Chuỗi không có dấu cách (không dùng `str(tuple)` nếu quên `.replace(" ", "")`)?
- [ ] Thử `n = 2` (`(1,2)`), `n = 4` (`((1,4),(2,3))`), `n = 8`?
- [ ] Skeleton đúng `class Solution` / `def findContestMatch(self, n: int) -> str`?